# Watson Healthcare Employee Attrition -- Data Cleaning & Exploration
## Purpose: Clean and prepare raw employee data for SQL analysis and Tableau visualization

In [3]:
import pandas as pd

df = pd.read_csv('watson_healthcare_modified2.csv')
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
df.head()

Rows: 1676
Columns: 35


,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,RelationshipSatisfaction,StandardHours,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1313919,41,No,Travel_Rarely,1102,Cardiology,1,2,Life Sciences,1,...,1,80,0,8,0,1,6,4,0,5
1,1200302,49,No,Travel_Frequently,279,Maternity,8,1,Life Sciences,1,...,4,80,1,10,3,3,10,7,1,7
2,1060315,37,Yes,Travel_Rarely,1373,Maternity,2,2,Other,1,...,2,80,0,7,3,3,0,0,0,0
3,1272912,33,No,Travel_Frequently,1392,Maternity,3,4,Life Sciences,1,...,3,80,0,8,3,3,8,7,3,0
4,1414939,27,No,Travel_Rarely,591,Maternity,2,1,Medical,1,...,4,80,1,6,3,3,2,2,2,2


## Step 1: Initial Data Inspection

In [4]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1676 entries, 0 to 1675
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   EmployeeID                1676 non-null   int64 
 1   Age                       1676 non-null   int64 
 2   Attrition                 1676 non-null   object
 3   BusinessTravel            1676 non-null   object
 4   DailyRate                 1676 non-null   int64 
 5   Department                1676 non-null   object
 6   DistanceFromHome          1676 non-null   int64 
 7   Education                 1676 non-null   int64 
 8   EducationField            1676 non-null   object
 9   EmployeeCount             1676 non-null   int64 
 10  EnvironmentSatisfaction   1676 non-null   int64 
 11  Gender                    1676 non-null   object
 12  HourlyRate                1676 non-null   int64 
 13  JobInvolvement            1676 non-null   int64 
 14  JobLevel                

In [5]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate EmployeeIDs: {df['EmployeeID'].duplicated().sum()}")

Duplicate rows: 0
Duplicate EmployeeIDs: 0


In [6]:
# Check unique values for categorical columns
for col in ['Attrition', 'Department', 'Gender', 'JobRole', 'OverTime', 'MaritalStatus', 'BusinessTravel', 'EducationField']:
    print(f"{col}: {df[col].unique()}")
    print()

Attrition: ['No' 'Yes']

Department: ['Cardiology' 'Maternity' 'Neurology']

Gender: ['Female' 'Male']

JobRole: ['Nurse' 'Other' 'Therapist' 'Administrative' 'Admin']

OverTime: ['Yes' 'No']

MaritalStatus: ['Single' 'Married' 'Divorced']

BusinessTravel: ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']

EducationField: ['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']



In [8]:
print(f"Before: {df['JobRole'].value_counts().to_dict()}")

df['JobRole'] = df['JobRole'].replace('Admin', 'Administrative')

print(f"After: {df['JobRole'].value_counts().to_dict()}")

Before: {'Nurse': 822, 'Other': 534, 'Therapist': 189, 'Administrative': 115, 'Admin': 16}
After: {'Nurse': 822, 'Other': 534, 'Therapist': 189, 'Administrative': 131}


### 2.2 Map Education levels to labels
Education column uses numbers 1-5. Mapping to readable labels for analysis and visualization.z

In [9]:
education_map = {
    1: 'Below College',
    2: 'College',
    3: "Bachelor's",
    4: "Master's",
    5: 'Doctor'
}

df['EducationLevel'] = df['Education'].map(education_map)
print(df['EducationLevel'].value_counts())

EducationLevel
Bachelor's       655
Master's         447
College          322
Below College    196
Doctor            56
Name: count, dtype: int64


### 2.3 Map satisfaction and work-life balance ratings to labels

In [10]:
satisfaction_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}

df['JobSatisfactionLevel'] = df['JobSatisfaction'].map(satisfaction_map)
df['EnvironmentSatisfactionLevel'] = df['EnvironmentSatisfaction'].map(satisfaction_map)
df['WorkLifeBalanceLevel'] = df['WorkLifeBalance'].map(satisfaction_map)

print("Job Satisfaction:")
print(df['JobSatisfactionLevel'].value_counts())
print("\nWork-Life Balance:")
print(df['WorkLifeBalanceLevel'].value_counts())

Job Satisfaction:
JobSatisfactionLevel
Very High    530
High         507
Low          329
Medium       310
Name: count, dtype: int64

Work-Life Balance:
WorkLifeBalanceLevel
High         1028
Medium        385
Very High     173
Low            90
Name: count, dtype: int64


### 2.4 Create analytical groupings
Creating age, salary, tenure, and distance brackets for easier analysis and visualization.

In [11]:
# Age brackets
df['AgeBracket'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 65], 
                          labels=['18-25', '26-35', '36-45', '46-55', '56-65'])

# Salary brackets
df['SalaryBracket'] = pd.cut(df['MonthlyIncome'], bins=[0, 3000, 6000, 10000, 20000],
                              labels=['Low (<3K)', 'Mid (3K-6K)', 'High (6K-10K)', 'Very High (10K+)'])

# Tenure brackets
df['TenureBracket'] = pd.cut(df['YearsAtCompany'], bins=[-1, 2, 5, 10, 50],
                              labels=['0-2 years', '3-5 years', '6-10 years', '10+ years'])

# Distance brackets
df['DistanceBracket'] = pd.cut(df['DistanceFromHome'], bins=[-1, 5, 10, 20, 50],
                                labels=['0-5 miles', '6-10 miles', '11-20 miles', '21+ miles'])

print("Age:\n", df['AgeBracket'].value_counts().sort_index())
print("\nSalary:\n", df['SalaryBracket'].value_counts().sort_index())
print("\nTenure:\n", df['TenureBracket'].value_counts().sort_index())
print("\nDistance:\n", df['DistanceBracket'].value_counts().sort_index())

Age:
 AgeBracket
18-25    139
26-35    694
36-45    530
46-55    262
56-65     51
Name: count, dtype: int64

Salary:
 SalaryBracket
Low (<3K)           446
Mid (3K-6K)         597
High (6K-10K)       308
Very High (10K+)    325
Name: count, dtype: int64

Tenure:
 TenureBracket
0-2 years     387
3-5 years     491
6-10 years    514
10+ years     284
Name: count, dtype: int64

Distance:
 DistanceBracket
0-5 miles      721
6-10 miles     448
11-20 miles    269
21+ miles      238
Name: count, dtype: int64


## Step 3: Outlier Detection


In [12]:
numeric_cols = ['Age', 'MonthlyIncome', 'DailyRate', 'HourlyRate', 'DistanceFromHome', 
                'YearsAtCompany', 'TotalWorkingYears', 'YearsInCurrentRole']

df[numeric_cols].describe().round(2)

,Age,MonthlyIncome,DailyRate,HourlyRate,DistanceFromHome,YearsAtCompany,TotalWorkingYears,YearsInCurrentRole
count,1676.00,1676.00,1676.00,1676.00,1676.00,1676.00,1676.00,1676.00
mean,36.87,6516.51,800.56,65.47,9.22,7.03,11.34,4.26
std,9.13,4728.46,401.59,20.21,8.16,6.10,7.83,3.63
min,18.00,1009.00,102.00,30.00,1.00,0.00,0.00,0.00
25%,30.00,2928.25,465.00,48.00,2.00,3.00,6.00,2.00
50%,36.00,4899.00,796.50,65.50,7.00,5.00,10.00,3.00
75%,43.00,8380.25,1157.00,83.00,14.00,10.00,15.00,7.00
max,60.00,19999.00,1499.00,100.00,29.00,40.00,40.00,18.00


## Step 4: Export Clean Dataset
Exporting the cleaned and enriched dataset for SQL analysis and Tableau visualization.

In [14]:
print(f"Original columns: 35")
print(f"Final columns: {len(df.columns)}")
print(f"New columns added: {len(df.columns) - 35}")
print(f"\nNew columns: EducationLevel, JobSatisfactionLevel, EnvironmentSatisfactionLevel, WorkLifeBalanceLevel, AgeBracket, SalaryBracket, TenureBracket, DistanceBracket")
print(f"\nCleaning applied:")
print(f"- Merged 'Admin' into 'Administrative' (16 rows)")
print(f"- No nulls found")
print(f"- No duplicates found")
print(f"- No outliers detected")

df.to_csv('watson_healthcare_cleaned.csv', index=False)
print(f"\nExported: watson_healthcare_cleaned.csv ({len(df)} rows, {len(df.columns)} columns)")

Original columns: 35
Final columns: 43
New columns added: 8

New columns: EducationLevel, JobSatisfactionLevel, EnvironmentSatisfactionLevel, WorkLifeBalanceLevel, AgeBracket, SalaryBracket, TenureBracket, DistanceBracket

Cleaning applied:
- Merged 'Admin' into 'Administrative' (16 rows)
- No nulls found
- No duplicates found
- No outliers detected

Exported: watson_healthcare_cleaned.csv (1676 rows, 43 columns)
